In [ ]:
# ルートに移動

%cd ..

In [ ]:
# ライブラリのインポート

import glob
import os
import shutil
import yaml
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO
from ultralytics.utils.ops import xywhn2xyxy
from ultralytics.utils.metrics import ConfusionMatrix

In [ ]:
# ディレクトリの定義

audited_dir = os.path.join("data", "audited")
splited_dir = os.path.join("data", "splited")

In [ ]:
# モデルの読み込み

n_splits = 5

models = []

for i in range(1, n_splits + 1):
    model_path = os.path.join("runs", "exp", f"split_{i}", "weights", "best.pt")

    if os.path.exists(model_path):
        print(model_path)
        model = YOLO(model_path)
        models.append(model)

In [ ]:
# モデルの監査

matrix = None

for i in range(1, n_splits + 1):
    annotation_path = os.path.join(splited_dir, f"split_{i}", "data.yaml")
    with open(annotation_path, "r") as f:
        data = yaml.safe_load(f)

    names = data["names"]

    cm = ConfusionMatrix(task="detect", names=names, save_matches=True)

    images = sorted(
        glob.glob(os.path.join(splited_dir, f"split_{i}", "val", "images", "*.jpg")),
        key=lambda p: int(os.path.basename(p)[1:-4]),
    )

    labels = sorted(
        glob.glob(os.path.join(splited_dir, f"split_{i}", "val", "labels", "*.txt")),
        key=lambda p: int(os.path.basename(p)[1:-4]),
    )

    for image, label in zip(images, labels):
        image_rgb = Image.open(image).convert("RGB")

        result = np.loadtxt(label, dtype=np.float32, ndmin=2)

        xywh = torch.from_numpy(result[:, 1:5])
        xyxy = xywhn2xyxy(xywh, w=image_rgb.width, h=image_rgb.height)
        gt_boxes = xyxy.float()

        gt_classes = torch.from_numpy(result[:, 0]).to(torch.int64)

        results = model.predict(
            source=np.array(image_rgb),
            half=True,
            conf=0.2,
            iou=0.5,
            max_det=5,
            verbose=False,
        )

        if len(results[0].boxes) == 0:
            pd_boxes = torch.empty((0, 4), dtype=torch.float32)
            pd_scores = torch.empty((0,), dtype=torch.float32)
            pd_classes = torch.empty((0,), dtype=torch.int64)
        else:
            pd_boxes = results[0].boxes.xyxy.cpu().float()
            pd_scores = results[0].boxes.conf.cpu().float()
            pd_classes = results[0].boxes.cls.cpu().to(torch.int64)

        detections = {
            "bboxes": pd_boxes,
            "cls": pd_classes,
            "conf": pd_scores,
        }

        batch = {"bboxes": gt_boxes, "cls": gt_classes}
        cm.process_batch(detections, batch, conf=0.2, iou_thres=0.5)

        fp = cm.matches["FP"]["bboxes"]
        fn = cm.matches["FN"]["bboxes"]
        
        if len(fp) > 0 or len(fn) > 0:
            image_tensor = (
                torch.from_numpy(np.array(image_rgb))
                .permute(2, 0, 1)
                .float()
                .unsqueeze(0)
                / 255.0
            )
            file_name = os.path.basename(image)
            cm.plot_matches(image_tensor, file_name, save_dir=Path(audited_dir, f"split_{i}"))

    src = os.path.join(audited_dir, f"split_{i}", "visualizations")
    dst = os.path.join(audited_dir, f"split_{i}")
    shutil.copytree(src, dst, dirs_exist_ok=True)
    shutil.rmtree(src)

    if matrix is None:
        matrix = cm.matrix.astype(np.int64).copy()
    else:
        assert matrix.shape == cm.matrix.shape
        matrix += cm.matrix.astype(np.int64)

np.save(os.path.join(audited_dir, "confusion_matrix.npy"), matrix.astype(np.int32))